In [ ]:
from __future__ import annotations

import os
import openai
import json
import pandas as pd

from pathlib import Path
from dotenv import load_dotenv
from openai import AzureOpenAI

import tiktoken

load_dotenv(Path.cwd() / "OPENAI.env")
endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
version = os.getenv("AZURE_OPENAI_API_VERSION")
model = os.getenv("GENEAGENT_MODEL")
reasoning = os.getenv("OPENAI_REASONING_EFFORT")

client = AzureOpenAI(
    api_version = version,
    azure_endpoint = endpoint,
    api_key = api_key,
)


try:
    encoding = tiktoken.encoding_for_model("gpt-5")
except KeyError:
    encoding = tiktoken.get_encoding("o200k_base")
    
MAX_CONTEXT_TOKENS = 128000          # total (prompt + completion)
RESERVED_COMPLETION_TOKENS = 24000    # how many tokens you want to allow for the answer
SAFETY_MARGIN_TOKENS = 4000          # to cover system message + role overhead, etc.

MAX_PROMPT_TOKENS = (
    MAX_CONTEXT_TOKENS
    - RESERVED_COMPLETION_TOKENS
    - SAFETY_MARGIN_TOKENS
)


In [ ]:
def llm_reasoning(prompt):
    system, narrow_prompt = prompt[0], prompt[1]

    try:
        response = client.responses.create(
            model="gpt-5.2",
            instructions=system,
            input=narrow_prompt,
            reasoning={"effort": "high"},
            max_output_tokens=RESERVED_COMPLETION_TOKENS,
        )

        raw_text = response.output_text

        if raw_text is None or raw_text.strip() == "":
            return (
                "ERROR: Model returned no visible text. "
                "Try increasing max_output_tokens or lowering reasoning effort."
            )

        cleaned = (
            raw_text
            .replace("```json", "")
            .replace("```plaintext", "")
            .replace("```", "")
            .strip()
        )

        return cleaned

    except Exception as e:
        return f"ERROR in Generating Response: {type(e).__name__}: {e}"

# Generate the Enrichment Analysis Chains

In [ ]:
example = pd.read_csv("data/case.test.tsv", header=0, index_col='PMC', sep="\t")

with open("library/StdEnrich.json","r") as rfile:
    lib = json.load(rfile)

In [ ]:
def truncate_to_tokens(text, max_tokens) :
    """Truncate `text` so that it uses at most `max_tokens` tokens."""
    tokens = encoding.encode(text)
    if len(tokens) <= max_tokens:
        return text
    return encoding.decode(tokens[:max_tokens])

def get_prompt_chains(background: str, description: dict) -> tuple:
    system = f"""You are an expert biomedical research assistant specialized in enrichment analysis, biological databases, omics interpretation, and experimental design. 
Your task is to analyze a research context and a list of candidate enrichment databases, then select databases that best match the research goal to construct logical enrichment analysis chains. 
You must use only the information provided in the input. Do not invent database functions, hidden capabilities, biological annotations, species coverage, supported input types, or external facts. If information is unclear or missing, explicitly state the uncertainty."""
    
    task = f"""Strictly follow the step-by-step task:
1. Interpret the research context.
2. Evaluate every candidate database against the research goal and the database’s own functional description.
3. Classify each database as Strong Match, Moderate Match, Possible Match, or Not Matched.
4. Select the STRONG and MODERATE matched databases.
5. Construct several logical enrichment analysis chains in the form Database A -> Database B -> Database C.
6. Recommend one primary chain by explaining the purpose, step-by-step logic, transition justification, confidence, and uncertainty for each chain.
7. Provide alternative chains for different analytical goals.
8. Do not hallucinate or infer unsupported database capabilities.

Evaluation criteria:
- Context relevance
- Input compatibility
- Functional specificity
- Complementarity with other selected databases
- Utility in a logical enrichment analysis chain

Matching categories:
- Strong Match:
The database description clearly aligns with the research context, input type, and analysis goal.
- Moderate Match:
The database is relevant but partial, secondary, or missing some details.
- Possible Match:
The database may be useful, but the description is incomplete or compatibility is uncertain.
- Not Matched:
The database does not support the research goal or lacks relevant functionality.

Chain construction rules:
- Use only Strong and Moderate Match databases.
- Do not use Possible and Not Matched databases in chains.
- Each chain must contain at least 2 databases.
- Prefer 3–5 databases per chain when enough matched databases exist.
- Each chain must have a clear analytical theme.
- Each transition must be justified.
- Do not imply direct software interoperability unless explicitly stated.

Important constraints:
- Do not hallucinate.
- Do not introduce databases not present in the candidate list.
- Do not claim unsupported database capabilities.
- Do not over-select weakly relevant databases.
- Do not create random chains.
- Every selected database and every chain transition must be justified using the provided context and descriptions.
- If the input is insufficient, explain what is missing and provide the best possible cautious output.
"""
    
    style = """
Return your answer as valid JSON only. Do not include markdown.
Use this schema:
{
    "selected_databases": [
        {
        "database": "",
        "match_level": "",
        "role_in_analysis": "",
        "why_selected": "",
        }
    ],
    "enrichment_analysis_chains": [
        {
        "chain_name": "",
        "chain": ["Database A", "Database B", "Database C"],
        "purpose": "",
        "step_by_step_logic": [
            {
            "database": "",
            "role": ""
            }
        ],
        "transition_justification": [
            {
            "from": "",
            "to": "",
            "justification": ""
            }
        ],
        "confidence": "High | Medium | Low",
        }
    ],
    "recommended_primary_chain": {
        "chain": ["Database A", "Database B", "Database C"],
        "reason": "",
        "essential_databases": [
        {
            "database": "",
            "reason": ""
        }
        ],
        "optional_databases": [
        {
            "database": "",
            "reason": ""
        }
        ]
    },
    "alternative_chains": [
        {
        "goal": "",
        "chain": ["Database A", "Database B", "Database C"],
        "reason": ""
        }
    ]
}
"""
    parameter = f"""
        Research Background/Context: {background}
        Funcaltional Document of Candidate Enrichment Dabases: {description}
        """
    prompt = task.strip() + style.strip() + parameter.strip()
    narrow_prompt = truncate_to_tokens(prompt, MAX_PROMPT_TOKENS)
    
    return (system, narrow_prompt)

In [ ]:
def search_databases(lib, category):
    matched = {}
    for key, value in lib.items():
        if value["Category"] == category:
            function = value["Overall functional summary"] + " " + value["What the gene sets represent"] + " " + value["Important interpretation"]
            matched[key] = function
    return matched

for goal, category, experiment, species in zip(example["Research Objectives"], example["Database"], example["Experimental Conditions"], example["Species"]):
    bg = f"Overall research goal: {goal}. \nExperimental requirements for gene sets: {experiment}"
    databases = search_databases(lib, category)
    print(databases)
    prompt = get_prompt_chains(bg, databases)
    print(prompt)
    chains = llm_reasoning(prompt)
    print(chains)
    

In [ ]:
chains_json = json.loads(chains)
chains_json["enrichment_analysis_chains"]

# Get the Enrichment Functions and Genes from Enrichr

In [ ]:
from textwrap import indent

from apis.utils import dumps, request_json


BACKGROUND_TYPES = [
    'TISSUES_Curated_2025',
    'GO_Biological_Process_2026',
    'MGI_Mammalian_Phenotype_Level_4_2024'
   ]


def get_pathway_for_gene_set(gene_set, threshold=0.01):
    """Return top pathway/enrichment terms from Enrichr."""

    gene_list = [gene.strip() for gene in gene_set.split(",") if gene.strip()]
    if not gene_list:
        return dumps({})

    payload = {
        "list": (None, "\n".join(gene_list)),
        "description": (None, "GeneAgent gene set"),
    }
    data, error = request_json(
        "POST",
        "https://maayanlab.cloud/Enrichr/addList",
        files=payload,
    )
    if error:
        return {"ERROR": error}
    if not isinstance(data, dict) or "userListId" not in data:
        return {"ERROR": "Error: unexpected Enrichr addList response format"}

    list_id = data["userListId"]
    terms = {}
    for background_type in BACKGROUND_TYPES:
        terms[background_type] = [] 
        results, result_error = request_json(
            "GET",
            "https://maayanlab.cloud/Enrichr/enrich",
            params={"userListId": list_id, "backgroundType": background_type},
        )
        if result_error or not isinstance(results, dict):
            continue

        pathway_data = results.get(background_type, [])
        if not isinstance(pathway_data, list):
            continue

        for value in pathway_data:
            if not isinstance(value, list) or len(value) < 6:
                continue
            p_value = value[2]
            if p_value > threshold:
                continue
            
            term_name = value[1]
            overlapping = ",".join(value[5]) if isinstance(value[5], list) else []
            temp = {
                "term": term_name,
                "p-value": p_value,
                "overlapping genes": overlapping,
            }
            terms[background_type].append(temp)

    return terms

gene_set = example["DEGs"].to_list()[0]
enriched_results = get_pathway_for_gene_set(gene_set)
print(type(enriched_results))


In [81]:
enriched_results

{'TISSUES_Curated_2025': [{'term': 'PLASMA CELL',
   'p-value': 1.2961851482695205e-10,
   'overlapping genes': 'NRP1,C1QA,CSF1R,IL1RN,ECM1,CFH,PLXND1,PROS1,ITGB3,PTPRM,F13A1,C4B,ALCAM,ADAMTSL4,ANPEP,LIPG,CD163,VCAM1,RNASE4,MERTK,LYVE1,CEACAM1,SELL,CD109,SELENOP,CFB,CD44,PCYOX1,NECTIN1'},
  {'term': 'BONE MARROW CELL',
   'p-value': 1.6704449450538065e-10,
   'overlapping genes': 'NRP1,C1QA,CSF1R,IL1RN,ECM1,CFH,PLXND1,PROS1,ITGB3,PTPRM,F13A1,C4B,ALCAM,ADAMTSL4,ANPEP,LIPG,CD163,VCAM1,RNASE4,MERTK,LYVE1,CEACAM1,SELL,CD109,SELENOP,CFB,CD44,PCYOX1,NECTIN1'},
  {'term': 'MARROW CELL',
   'p-value': 2.24229106320072e-10,
   'overlapping genes': 'C1QA,CXCL9,SERPINB2,LY86,CXCR4,PPBP,CXCL3,DCSTAMP,HPGDS,CEACAM1,IL1B,TNIP3,GPR171,CD300E,SIGLEC1,CD44,C1QC'},
  {'term': 'MONONUCLEAR PHAGOCYTE',
   'p-value': 5.2945147943563e-10,
   'overlapping genes': 'C1QA,CXCL9,SERPINB2,LY86,CXCR4,PPBP,CXCL3,DCSTAMP,CEACAM1,IL1B,TNIP3,GPR171,CD300E,SIGLEC1,C1QC'},
  {'term': 'MONOCYTE',
   'p-value': 5.29451479

# Filer the enrichment results based on context

In [ ]:
def truncate_to_tokens(text, max_tokens) :
    """Truncate `text` so that it uses at most `max_tokens` tokens."""
    tokens = encoding.encode(text)
    if len(tokens) <= max_tokens:
        return text
    return encoding.decode(tokens[:max_tokens])

def get_prompt_filter(background: str, database: str, candidate: list) -> tuple:
    system = f"""You are an expert biomedical research assistant specializing in enrichment analysis interpretation, biological context matching, and evidence-based result filtering.
    Your task is to filter enrichment results obtained from the selected databases according to a given research background context.
    You must retain only enriched terms that are strongly relevant to the provided research context.
    You must not change, rewrite, normalize, abbreviate, expand, merge, split, translate, correct, or rename any enriched term. Every retained enriched term must appear exactly as provided in the input."""
    
    task = f"""Strictly follow the step-by-step task:
    1. Parse the research background context and identify the key contextual anchors:
        - Disease or phenotype
        - Biological process or mechanism
        - Tissue, organ, or cell type
        - Experimental condition
        - Species
        - Data type
        - Desired interpretation focus
        - Exclusion criteria
    2. Evaluate every enriched term from the enrichment results.
    3. Classify each enriched term as:
        - Strong Retain
        - Conditional Retain
        - Exclude
    4. By default, retain only Strong Retain terms.
    5. If there is no Strong Retain terms or user explicitly asks for broader recall, include Conditional Retain terms in the final retained output, clearly labeled.

    Filtering criteria:
        Retain an enriched term only if it is directly and strongly relevant to at least one major contextual anchor in the research background.
        Do not introduce any enriched term that is not present in the input.
        Do not modify any enriched term text.
        Do not infer unsupported database capabilities or biological relationships.
        If statistical metadata is provided, preserve it exactly as given and do not recalculate or modify it.
        
    Exclude terms that are:
        - Too generic
        - Weakly related
        - Ambiguous
        - Outside the disease, tissue, cell type, organism, mechanism, or experimental context
        - Redundant broad parent terms when more specific relevant terms are available
        - Unsupported by the provided context
        
    Critical preservation rule:
        For every retained term, copy the enriched term exactly from the input. Preserve spelling, capitalization, punctuation, spacing, and database-provided wording. Do not rewrite terms.

    Before finalizing the answer, perform an integrity check:
        1. Every term in final_retained_term_list must exactly match one term from the input enrichment results.
        2. No retained term may contain edited wording.
        3. No term may be added from reasoning or external knowledge.
        4. If an exact match cannot be verified, remove the term from the final retained list.
        """
    
    style = """
    Return your answer as valid JSON only. Do not include markdown.
    Use this schema:
    {
        "filtering_summary": {
            "input_enriched_term_count": 0,
            "strong_retain_count": 0,
            "conditional_retain_count": 0,
            "exclude_count": 0,
            "uncertain_count": 0,
            "main_contextual_criteria": [],
            "important_uncertainty": []
        },
        "retained_enriched_terms": [
            {
            "database": "",
            "enriched_term": "",
            "retention_level": "Strong Retain | Conditional Retain",
            "contextual_relevance_reason": "",
            }
        ],
        "final_retained_term_list": [
            ""
        ]
    }
    """
    parameter = f"""
    Research Background/Context: {background}
    Enrichment Database: {database}
    Candidate Enrichment Terms: {candidate}
    """
    prompt = task.strip() + style.strip() + parameter.strip()
    narrow_prompt = truncate_to_tokens(prompt, MAX_PROMPT_TOKENS)
    
    return (system, narrow_prompt)

In [ ]:
filtered_terms = {}

for key, value in enriched_results.items():
    print(key)
    candidate = []
    for in_dict in value:
        candidate.append(in_dict["term"])
    print(len(candidate))
    prompt_filter = get_prompt_filter(bg, key, candidate)
    filter = llm_reasoning(prompt_filter)
    print(filter)
    filtered_terms[key] = json.loads(filter)

print(len(filtered_terms))

In [73]:
filtered_terms

{'TISSUES_Curated_2025': {'filtering_summary': {'input_enriched_term_count': 36,
   'strong_retain_count': 6,
   'conditional_retain_count': 6,
   'exclude_count': 24,
   'uncertain_count': 0,
   'main_contextual_criteria': ['Tumor-associated macrophages (TAMs) / macrophage-lineage myeloid cells',
    'Myeloid-specific PD-1 (Pdcd1) ablation (LysMCre) context',
    'Differential expression measured in TAMs from Pdcd1 f/f LysMCre vs Pdcd1 f/f mice',
    'Antitumor immune response focus restricted to TAM-relevant myeloid/hematopoietic compartments',
    'Prefer specific myeloid/macrophage/hematopoietic terms over generic organismal, developmental, non-immune, or unrelated tissue/cancer-cell terms'],
   'important_uncertainty': ['Broad immune/hematopoietic and blood compartment terms (e.g., IMMUNE SYSTEM, HEMATOPOIETIC SYSTEM, BLOOD) are relevant to myeloid origin but are less specific to TAMs; treated as Conditional Retain and not included by default.']},
  'retained_enriched_terms': [{'d

# Cluster the Enrichment terms

In [ ]:
def truncate_to_tokens(text, max_tokens) :
    """Truncate `text` so that it uses at most `max_tokens` tokens."""
    tokens = encoding.encode(text)
    if len(tokens) <= max_tokens:
        return text
    return encoding.decode(tokens[:max_tokens])

def get_prompt_cluster(background: str, chain: str, filter: list) -> tuple:
    system = f"""You are an expert biomedical research assistant specializing in enrichment analysis interpretation, biological pathway reasoning, disease mechanism analysis, and multi-database functional integration.
    Your task is to categorize filtered enrichment terms from multiple enrichment databases into distinct clusters based on their intrinsic functional coherence, biological relatedness, or mechanistic synergy.
    The clusters must also be informed by the provided enrichment analysis chains. Each cluster should represent a coherent biological interpretation module that helps answer the research goal or experimental requirements described in the research background.
    Clusters may overlap. The same enrichment term may appear in multiple clusters if it supports multiple biological interpretations. A cluster may contain only one enrichment term if that term independently represents a distinct and important biological module.
    The final output must be valid JSON only."""
    
    task = f"""Strictly follow the step-by-step task:
    1. Parse the research background context and identify key contextual anchors.
    2. Parse the preliminary enrichment analysis chains and identify how they should guide cluster organization.
    3. Parse all filtered enrichment terms from all databases.
    4. Cluster the filtered terms based on:
        - Functional coherence
        - Biological relatedness
        - Mechanistic synergy
        - Cross-database complementarity
        - Relevance to the research goal
        - Consistency with the preliminary enrichment chains
    5. Allow clusters to share terms when biologically justified.
    6. Allow single-term clusters when a term represents a distinct important biological signal.
    7. Do not force every term into a cluster. Use unclustered_terms when coherence is unclear.
    8. Preserve every enriched term exactly as provided.
    9. Do not introduce new enriched terms.
    10. Do not modify metadata.
    11. Do not hallucinate biological functions, causal relationships, or database capabilities.
    12. Return valid JSON only.

    Critical preservation rule:
        Every clustered enriched term must exactly match one enriched term from the input. Preserve spelling, capitalization, punctuation, spacing, and database-provided wording. Do not rewrite, translate, normalize, abbreviate, expand, correct, merge, split, or replace enriched terms.
    
    Before finalizing the JSON, perform an internal integrity check:
        1. Every value in clusters[].terms[].enriched_term must exactly match one input enriched term.
        2. No enriched term may be rewritten, translated, normalized, corrected, merged, or split.
        3. No metadata value may be changed.
        4. No new enriched terms may be introduced.
        5. The JSON must be parseable by a standard JSON parser.
        6. If any term cannot be verified as an exact input term, remove it from clusters
    """
    
    style = """
    Return your answer as valid JSON only. Do not include markdown.
    Use this schema:
    {
    "clusters": [
        {
        "cluster_id": "C1",
        "cluster_name": "",
        "core_intent": "",
        "research_goal_connection": "",
        "related_preliminary_chain": "",
        "terms": [
            {
            "database": "",
            "enriched_term": "",
            "reason_for_inclusion": ""
            }
        ],
        "confidence": "High | Medium | Low",
        }
    ],
    "unclustered_terms": [
        {
        "database": "",
        "enriched_term": "",
        "reason_not_clustered": ""
        }
    ]
    }
    """
    parameter = f"""
    Research Background/Context: {background}
    Primary Enrichment Analysis Chain: {chain}
    Filtered Enrichment Terms: {filter}
    """
    prompt = task.strip() + style.strip() + parameter.strip()
    narrow_prompt = truncate_to_tokens(prompt, MAX_PROMPT_TOKENS)
    
    return (system, narrow_prompt)

In [79]:
enrichment_chain = "->".join(chains_json["recommended_primary_chain"]["chain"])
chain_explanation = chains_json["recommended_primary_chain"]["reason"]
chain_input = f"{enrichment_chain}: {chain_explanation}"

filter_input = []
for key in filtered_terms.keys():
    temp = {}
    temp["Database Name"] = key
    temp["Filtered Terms"] = filtered_terms[key]["final_retained_term_list"]
    filter_input.append(temp)
print(filter_input)

prompt_cluster = get_prompt_cluster(bg, chain_input, filter_input)
clustered_terms = json.loads(llm_reasoning(prompt_cluster))

[{'Database Name': 'TISSUES_Curated_2025', 'Filtered Terms': ['MONONUCLEAR PHAGOCYTE', 'MONOCYTE', 'PHAGOCYTE', 'BONE MARROW CELL', 'BONE MARROW', 'HEMATOPOIETIC CELL']}, {'Database Name': 'GO_Biological_Process_2026', 'Filtered Terms': ['Inflammatory Response (GO:0006954)', 'Response to Lipopolysaccharide (GO:0032496)', 'Cellular Response to Lipopolysaccharide (GO:0071222)', 'Regulation of Macrophage Migration (GO:1905521)', 'Cellular Response to Molecule of Bacterial Origin (GO:0071219)', 'Myeloid Leukocyte Differentiation (GO:0002573)', 'Regulation of Toll-Like Receptor 4 Signaling Pathway (GO:0034143)', 'Mononuclear Cell Differentiation (GO:1903131)', 'Positive Regulation of Nitric Oxide Biosynthetic Process (GO:0045429)', 'Regulation of Innate Immune Response (GO:0045088)', 'Regulation of Interleukin-6 Production (GO:0032675)', 'Regulation of Canonical NF-kappaB Signal Transduction (GO:0043122)', 'Positive Regulation of Phagocytosis (GO:0050766)', 'Regulation of Interleukin-1 Prod

# Match Gene Clusters

In [95]:
import numpy as np

gene_cluster = [["Cluster ID", "Gene List", "Gene Count"]]
for item in clustered_terms["clusters"]:
    genes = set()
    # p_value = []
    id = "Case_1-"+ item["cluster_id"]
    for in_term in item["terms"]:
        database = in_term["database"]
        enrich_term = in_term["enriched_term"]
        
        for value in enriched_results[database]:
            if value["term"].lower().strip() == enrich_term.lower().strip():
                # p_value.append(value["p-value"])
                genes = genes.union(set(value["overlapping genes"].split(",")))
                            
    # gene_cluster[item["cluster_id"]] = {"Gene Cluster": ",".join(list(genes)), "Avg. P-value": np.average(p_value)}
    gene_cluster.append([id, ",".join(list(genes)), len(list(genes))])

print(len(gene_cluster))
np.savetxt("data/gene.cluster.tsv", np.array(gene_cluster), fmt="%s", delimiter="\t", newline="\n")

10


In [ ]:
filtered_terms

In [ ]:
post = {}
for key in enriched_results.keys():
    post[key] = {}
    for in_term in enriched_results[key]:
        genes = in_term["overlapping genes"]
        name = in_term["term"]
        if genes in post[key]:
            post[key][genes].append(name)
        else:
            post[key][genes] = [name]
        
print(post)        